### Cellule 1 - Installer boto3 dans Jupyter :

In [4]:
# L'image jupyter/scipy-notebook ne contient pas boto3 par défaut.
# On l'installe dans le kernel courant.
%pip install boto3==1.34.0

Note: you may need to restart the kernel to use updated packages.


### Cellule 2 - Connexion à MinIO :

In [5]:
import boto3
# Note importante : depuis le conteneur Jupyter, MinIO est accessible
# par son nom de service "minio", PAS par "localhost".
# C'est la résolution DNS automatique du réseau Compose.
s3 = boto3.client(
"s3",
endpoint_url="http://minio:9000",
aws_access_key_id="anfa-app-key",
aws_secret_access_key="anfa-app-secret-2026",
region_name="us-east-1",
)
# Vérifier la liste des buckets
s3.list_buckets()

{'ResponseMetadata': {'RequestId': '18BB3D7DAD94ADFB',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '366',
   'content-type': 'application/xml',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18BB3D7DAD94ADFB',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '2050',
   'x-ratelimit-remaining': '2050',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 21 Jun 2026 23:48:07 GMT'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'anfa-raw',
   'CreationDate': datetime.datetime(2026, 6, 21, 23, 3, 23, 190000, tzinfo=tzlocal())}],
 'Owner': {'DisplayName': 'minio',
  'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}

### Cellule 3 - Lister les objets du bucket :

In [7]:
reponse = s3.list_objects_v2(Bucket="anfa-raw", Prefix="referentiel/")
for obj in reponse.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size']} octets)")

referentiel/arrets.csv  (3821 octets)
referentiel/bus.csv  (5912 octets)
referentiel/lignes.csv  (945 octets)
referentiel/tarifs.csv  (718 octets)


### Cellule 4 - Lire un CSV directement depuis MinIO avec pandas :

In [8]:
import pandas as pd
from io import BytesIO
# Télécharger le CSV des lignes en mémoire
obj = s3.get_object(Bucket="anfa-raw", Key="referentiel/lignes.csv")
df_lignes = pd.read_csv(BytesIO(obj["Body"].read()))
df_lignes

,ligne_id,nom,terminus_depart,terminus_arrivee,nb_arrets,distance_km
0,L01,Adidogomé - Adawlato,Adidogomé Assiyéyé,Adawlato Marché,9,9.89
1,L02,Agoè Assiyéyé - Grand Marché,Agoè Assiyéyé Terminus,Adawlato Grand Marché,9,16.28
2,L03,Avédji - Adawlato,Avédji Limousine,Adawlato Marché,9,11.67
3,L04,Université de Lomé - Adawlato,Campus Universitaire,Adawlato Marché,8,5.65
4,L05,Hédzranawoé - Adawlato,Hédzranawoé Terminus,Adawlato Marché,8,7.37
5,L06,Bè - Tokoin Casablanca,Bè Kpota,Tokoin Casablanca,7,5.49
6,L07,Cacavéli - BIA Centre,Cacavéli Marché,BIA Centre,7,10.03
7,L08,Aéroport GTA - Adawlato,Aéroport GTA,Adawlato Marché,7,8.73
8,L09,Baguida - Adawlato,Baguida Terminus,Adawlato Marché,8,13.13
9,L10,Anfamé - Université de Lomé,Anfamé Terminus,Campus Universitaire,8,5.89


### Cellule 5 - Une petite analyse exploratoire :

In [9]:
# Top 3 des lignes les plus longues
df_lignes.nlargest(3, "distance_km")[["nom", "nb_arrets", "distance_km"]]

,nom,nb_arrets,distance_km
1,Agoè Assiyéyé - Grand Marché,9,16.28
8,Baguida - Adawlato,8,13.13
2,Avédji - Adawlato,9,11.67
